# UPI Thesis RAG Pipeline

Modular Retrieval-Augmented Generation pipeline for **Universitas Pendidikan Indonesia (UPI)** documents.

**6 independent stages** - each stage is its own section with reusable functions and reads/writes only
files on Google Drive, so any stage can run on its own (provided the previous stage's output exists).

| Stage | Section | Output |
|-------|---------|--------|
| 1 | Scraping / ingestion | `manifest.json` |
| 2 | Extraction (PDF text + OCR fallback) | `raw/*.json` |
| 3 | Cleaning | `cleaned/*.txt`, `cleaned/*.json` |
| 4 | Chunking | `chunked/chunks.jsonl` |
| 5 | Vector store (FAISS) | `index/` |
| 6 | RAG chain + evaluation | `eval/*.json` |

**Resumable** - every stage skips inputs whose output already exists. Reruns never duplicate data.
**Extensible** - add a new dataset folder by appending one line to `CONFIG["dataset_roots"]`.

## Cell 1 - Install dependencies

Run once per Colab session. Restart the runtime if Colab prompts you to.

In [ ]:
# Python 3.11+ / Google Colab compatible. Stable, widely-used libraries only.
!pip install -q \
    pymupdf==1.24.10 \
    pdfminer.six==20240706 \
    pytesseract==0.3.13 \
    pdf2image==1.17.0 \
    sentence-transformers==3.0.1 \
    faiss-cpu==1.8.0.post1 \
    langchain-text-splitters==0.3.2 \
    beautifulsoup4==4.12.3 \
    requests==2.32.3 \
    tqdm==4.66.5

# System packages for OCR (Tesseract) + PDF rasterisation (Poppler).
# Indonesian + English language packs for Tesseract.
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ind tesseract-ocr-eng poppler-utils

print("\nDependencies installed. If Colab asks, restart the runtime, then continue from Cell 2.")

## Cell 2 - Mount Drive, imports, logging

Mounts Google Drive and sets up logging + progress bars used by every stage.
Always run this cell (and Cell 3) first in a new session.

In [ ]:
import json
import re
import logging
import hashlib
import time
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, urlparse

from tqdm.auto import tqdm

# --- Mount Google Drive (safe to re-run; no-op if already mounted) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as e:  # running outside Colab
    IN_COLAB = False
    print(f"Not running in Colab ({e}). Drive paths must exist locally.")


# --- Logging: clean, timestamped, idempotent setup ---
def get_logger(name="rag"):
    """Return a configured logger. Safe to call many times (no duplicate handlers)."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter(
            "%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S"))
        logger.addHandler(handler)
    logger.propagate = False
    return logger


log = get_logger()
log.info("Imports ready. Drive mounted: %s", IN_COLAB)

## Cell 3 - Configuration

One `CONFIG` dictionary controls the whole pipeline. **To add a future dataset, append its path to
`dataset_roots` - nothing else is hardcoded.** All stage outputs live under `Dataset/_pipeline/`,
kept separate so each stage's results are independent and reloadable.

In [ ]:
# --------------------------------------------------------------------------
# CONFIG - edit here only. Add new dataset folders to `dataset_roots`.
# --------------------------------------------------------------------------
DRIVE_BASE = Path("/content/drive/MyDrive/Dataset")

CONFIG = {
    # Folders scanned recursively in Stage 1. Append new datasets here later.
    "dataset_roots": [
        DRIVE_BASE,
        DRIVE_BASE / "PPID",
        DRIVE_BASE / "LPPM_UPI",
        DRIVE_BASE / "Dataset_PMB_UPI",
        # e.g. DRIVE_BASE / "Akademik_2026",   <-- add future datasets like this
    ],

    # Pipeline output root. Each stage writes to its own subfolder.
    "pipeline_dir": DRIVE_BASE / "_pipeline",

    # Document types collected during ingestion.
    "doc_extensions": [".pdf", ".docx", ".doc", ".txt", ".html", ".htm", ".pptx", ".csv"],

    # --- Optional public crawl (Stage 1). Empty by default = no crawling. ---
    "crawl_seed_urls": [],          # e.g. ["https://ppid.upi.edu/"]
    "crawl_max_pages": 25,          # politeness cap
    "crawl_delay_seconds": 1.5,     # delay between requests (be a good citizen)
    "crawl_same_domain_only": True,

    # --- Extraction (Stage 2) ---
    "ocr_languages": "ind+eng",     # Tesseract: Indonesian + English
    "ocr_min_chars_per_page": 80,   # below this -> page treated as scanned -> OCR
    "ocr_dpi": 200,

    # --- Chunking (Stage 4) ---
    "chunk_size": 1000,             # characters
    "chunk_overlap": 200,

    # --- Vector store (Stage 5) ---
    # Multilingual model - strong on Indonesian + English.
    "embedding_model": "paraphrase-multilingual-MiniLM-L12-v2",
    "embedding_batch_size": 64,

    # --- RAG / evaluation (Stage 6) ---
    "top_k": 5,
}

# Derived output directories - created once, reused by every stage.
PIPE = CONFIG["pipeline_dir"]
DIRS = {
    "raw":      PIPE / "raw",        # Stage 2: extracted-but-not-cleaned text (JSON)
    "cleaned":  PIPE / "cleaned",    # Stage 3: cleaned text (.txt + .json)
    "chunked":  PIPE / "chunked",    # Stage 4: chunks.jsonl
    "index":    PIPE / "index",      # Stage 5: FAISS index + metadata
    "eval":     PIPE / "eval",       # Stage 6: retrieved contexts + answers + scores
    "logs":     PIPE / "logs",
}
MANIFEST_PATH = PIPE / "manifest.json"

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)


# --- Small shared helpers ---
def doc_id(source):
    """Stable short id for a document, derived from its source path/URL."""
    return hashlib.sha1(str(source).encode("utf-8")).hexdigest()[:16]


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def category_of(source_path):
    """Infer a category from the dataset folder a file lives in."""
    s = str(source_path)
    for root in CONFIG["dataset_roots"]:
        if s.startswith(str(root)) and root != DRIVE_BASE:
            return root.name
    return "Dataset_root"


log.info("Config loaded. Pipeline dir: %s", PIPE)
log.info("Scanning %d dataset roots.", len(CONFIG["dataset_roots"]))

## Cell 4 - Stage 1: Scraping / Ingestion

Recursively scans every folder in `CONFIG["dataset_roots"]`, collects supported document types, and
optionally crawls **public** UPI pages for document links (only if `crawl_seed_urls` is set).

- Pages blocked by Cloudflare or similar protection are **skipped and logged** - no bypass, no
  CAPTCHA solving, no anti-bot circumvention.
- Writes `manifest.json` with `source`, `title`, `type`, `category`, `status` for every item.
- Re-running merges with the existing manifest instead of duplicating entries.

In [ ]:
def scan_local_documents():
    """Recursively collect supported documents from all dataset roots."""
    exts = set(CONFIG["doc_extensions"])
    seen, items = set(), []
    for root in CONFIG["dataset_roots"]:
        root = Path(root)
        if not root.exists():
            log.warning("Dataset root missing, skipped: %s", root)
            continue
        for path in root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in exts:
                continue
            # Skip our own pipeline outputs.
            if str(path).startswith(str(CONFIG["pipeline_dir"])):
                continue
            key = str(path)
            if key in seen:
                continue
            seen.add(key)
            items.append({
                "id": doc_id(key),
                "source": key,
                "origin": "local",
                "title": path.stem,
                "type": path.suffix.lower().lstrip("."),
                "category": category_of(path),
                "status": "discovered",
            })
    return items


# --- Optional polite public crawl ------------------------------------------------
BLOCK_SIGNS = ("cloudflare", "captcha", "access denied", "are you human",
               "verify you are human", "attention required")


def _looks_blocked(resp):
    """Heuristic: is this response an anti-bot wall? If so we SKIP it (never bypass)."""
    if resp.status_code in (403, 429, 503):
        return True
    body = (resp.text or "")[:4000].lower()
    return any(sign in body for sign in BLOCK_SIGNS)


def crawl_public_pages():
    """Breadth-first crawl of public pages to discover document links.

    Respects: same-domain limit, page cap, request delay. Pages protected by
    anti-bot systems are skipped and logged. Does NOT attempt to defeat any
    protection, solve CAPTCHAs, or circumvent Cloudflare.
    """
    seeds = CONFIG["crawl_seed_urls"]
    if not seeds:
        log.info("No crawl seeds configured - skipping web crawl.")
        return []

    import requests
    from bs4 import BeautifulSoup

    exts = set(CONFIG["doc_extensions"])
    headers = {"User-Agent": "UPI-Thesis-RAG/1.0 (academic research)"}
    queue = list(seeds)
    visited, found = set(), []

    pbar = tqdm(total=CONFIG["crawl_max_pages"], desc="Crawling", unit="page")
    while queue and len(visited) < CONFIG["crawl_max_pages"]:
        url = queue.pop(0)
        if url in visited:
            continue
        visited.add(url)
        pbar.update(1)
        try:
            resp = requests.get(url, headers=headers, timeout=20)
        except Exception as e:
            log.warning("Request failed, skipped: %s (%s)", url, e)
            continue

        if _looks_blocked(resp):
            log.warning("Blocked by anti-bot protection, skipped: %s", url)
            found.append({"id": doc_id(url), "source": url, "origin": "web",
                           "title": url, "type": "page", "category": "web",
                           "status": "blocked_skipped"})
            continue
        if resp.status_code != 200 or "text/html" not in resp.headers.get("Content-Type", ""):
            continue

        soup = BeautifulSoup(resp.text, "html.parser")
        base_domain = urlparse(url).netloc
        for a in soup.find_all("a", href=True):
            link = urljoin(url, a["href"]).split("#")[0]
            suffix = Path(urlparse(link).path).suffix.lower()
            if suffix in exts:                       # a downloadable document
                found.append({
                    "id": doc_id(link), "source": link, "origin": "web",
                    "title": (a.get_text(strip=True) or Path(link).stem),
                    "type": suffix.lstrip("."), "category": "web",
                    "status": "discovered",
                })
            elif link.startswith("http") and link not in visited:
                if not CONFIG["crawl_same_domain_only"] or urlparse(link).netloc == base_domain:
                    queue.append(link)
        time.sleep(CONFIG["crawl_delay_seconds"])
    pbar.close()
    return found


def build_manifest(do_crawl=False):
    """Run ingestion and merge results into a persistent manifest (no duplicates)."""
    log.info("Stage 1: scanning local datasets...")
    items = scan_local_documents()
    if do_crawl:
        items += crawl_public_pages()

    existing = {row["id"]: row for row in read_json(MANIFEST_PATH, default=[])}
    for it in items:
        if it["id"] in existing:                     # keep prior status (e.g. extracted)
            existing[it["id"]].update({k: it[k] for k in ("title", "type", "category")})
        else:
            existing[it["id"]] = it
    manifest = list(existing.values())
    write_json(MANIFEST_PATH, manifest)

    by_status = {}
    for row in manifest:
        by_status[row["status"]] = by_status.get(row["status"], 0) + 1
    log.info("Manifest saved: %s (%d items)", MANIFEST_PATH, len(manifest))
    log.info("Status breakdown: %s", by_status)
    return manifest


# ---- RUN STAGE 1 ----
manifest = build_manifest(do_crawl=bool(CONFIG["crawl_seed_urls"]))

## Cell 5 - Stage 2: Extraction (PDF text + OCR fallback)

For each PDF: try fast text extraction with **PyMuPDF** first. If a page yields too little text
(scanned page), automatically fall back to **page-by-page OCR** (Tesseract, `ind+eng`). Page order
and per-page metadata are preserved. Non-PDF text formats are read directly.

Output: one JSON per document in `raw/`. Documents already extracted are skipped (resumable).

In [ ]:
import fitz  # PyMuPDF


def extract_pdf(path):
    """Extract a PDF page-by-page. OCR fallback for pages with too little text."""
    pages = []
    doc = fitz.open(path)
    for i, page in enumerate(doc):
        text = page.get_text("text").strip()
        method = "text"
        if len(text) < CONFIG["ocr_min_chars_per_page"]:
            # Page looks scanned - render to image and OCR it.
            try:
                import io
                import pytesseract
                from PIL import Image
                pix = page.get_pixmap(dpi=CONFIG["ocr_dpi"])
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                ocr_text = pytesseract.image_to_string(
                    img, lang=CONFIG["ocr_languages"]).strip()
                if len(ocr_text) > len(text):
                    text, method = ocr_text, "ocr"
            except Exception as e:
                log.warning("OCR failed on %s p%d: %s", path.name, i + 1, e)
        pages.append({"page": i + 1, "text": text, "method": method})
    doc.close()
    return pages


def extract_textlike(path):
    """Read .txt / .html / .htm as a single page."""
    raw = path.read_text(encoding="utf-8", errors="ignore")
    if path.suffix.lower() in (".html", ".htm"):
        from bs4 import BeautifulSoup
        raw = BeautifulSoup(raw, "html.parser").get_text("\n")
    return [{"page": 1, "text": raw.strip(), "method": "text"}]


def run_extraction():
    """Stage 2: extract every local manifest item; skip those already done."""
    manifest = read_json(MANIFEST_PATH, default=[])
    if not manifest:
        log.error("No manifest found. Run Stage 1 first.")
        return

    todo = [r for r in manifest if r["origin"] == "local"
            and not (DIRS["raw"] / f"{r['id']}.json").exists()]
    n_local = sum(1 for r in manifest if r["origin"] == "local")
    log.info("Stage 2: %d documents to extract (%d already done).",
             len(todo), n_local - len(todo))

    status = {r["id"]: r for r in manifest}
    for row in tqdm(todo, desc="Extracting", unit="doc"):
        path = Path(row["source"])
        try:
            if not path.exists():
                status[row["id"]]["status"] = "missing_file"
                continue
            ext = path.suffix.lower()
            if ext == ".pdf":
                pages = extract_pdf(path)
            elif ext in (".txt", ".html", ".htm"):
                pages = extract_textlike(path)
            else:
                # .docx/.pptx/.csv: recorded for future support; skip gracefully.
                status[row["id"]]["status"] = "unsupported_format"
                continue

            ocr_pages = sum(1 for p in pages if p["method"] == "ocr")
            write_json(DIRS["raw"] / f"{row['id']}.json", {
                "id": row["id"], "source": row["source"], "title": row["title"],
                "category": row["category"], "type": row["type"],
                "n_pages": len(pages), "ocr_pages": ocr_pages, "pages": pages,
                "extracted_at": datetime.now().isoformat(timespec="seconds"),
            })
            status[row["id"]]["status"] = "extracted"
        except Exception as e:
            log.warning("Extraction failed: %s (%s)", path.name, e)
            status[row["id"]]["status"] = "extract_error"

    write_json(MANIFEST_PATH, list(status.values()))
    log.info("Stage 2 complete. Raw JSON in: %s", DIRS["raw"])


# ---- RUN STAGE 2 ----
run_extraction()

## Cell 6 - Stage 3: Cleaning

Cleans extracted text: removes repeated headers/footers, page numbers, boilerplate, OCR noise and
excess whitespace, while **preserving** headings, bullet lists and paragraph structure.

- Repeated headers/footers are detected automatically (lines that recur across many pages).
- Output saved per document as **`.txt`** (human-readable) and **`.json`** (structured, page-aware).
- Documents already cleaned are skipped.

In [ ]:
PAGE_NUM_RE   = re.compile(r"^\s*(?:hal(?:aman)?\.?\s*)?[-]?\s*\d{1,4}\s*[-]?\s*$", re.I)
BULLET_RE     = re.compile(r"^\s*([\-\*]|\d+[\.\)]|[a-z][\.\)])\s+", re.I)
MULTISPACE_RE = re.compile(r"[ \t]{2,}")
OCR_NOISE_RE  = re.compile(r"(.)\1{4,}")           # 5+ repeated chars (OCR garbage)
HEADING_RE    = re.compile(r"^([A-Z0-9][A-Z0-9 .,:'\-/()]{3,80})$")


def _detect_repeated_lines(pages, min_ratio=0.4):
    """Find short lines that repeat across many pages - likely headers/footers."""
    from collections import Counter
    counts, n_pages = Counter(), max(len(pages), 1)
    for p in pages:
        for line in {ln.strip() for ln in p["text"].splitlines() if ln.strip()}:
            if len(line) <= 90:
                counts[line] += 1
    return {ln for ln, c in counts.items() if c / n_pages >= min_ratio and n_pages > 2}


def _is_heading(line):
    return bool(HEADING_RE.match(line.strip())) and len(line.split()) <= 12


def clean_page(text, repeated):
    """Clean one page while preserving headings, bullets and paragraphs."""
    out = []
    for line in text.splitlines():
        s = line.strip()
        if not s:
            out.append("")
            continue
        if s in repeated or PAGE_NUM_RE.match(s):
            continue                                  # drop header/footer/page number
        s = OCR_NOISE_RE.sub(r"\1", s)                # collapse OCR noise runs
        s = MULTISPACE_RE.sub(" ", s)
        if BULLET_RE.match(s):
            out.append("- " + BULLET_RE.sub("", s).strip())   # normalise bullet
        elif _is_heading(s):
            out.append("\n## " + s + "\n")            # mark heading
        else:
            out.append(s)
    # Collapse 3+ blank lines into a paragraph break.
    cleaned = re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()
    return cleaned


def run_cleaning():
    """Stage 3: clean every extracted document; skip those already cleaned."""
    raw_files = sorted(DIRS["raw"].glob("*.json"))
    if not raw_files:
        log.error("No extracted files found. Run Stage 2 first.")
        return

    todo = [f for f in raw_files if not (DIRS["cleaned"] / f"{f.stem}.json").exists()]
    log.info("Stage 3: %d documents to clean (%d already done).",
             len(todo), len(raw_files) - len(todo))

    for f in tqdm(todo, desc="Cleaning", unit="doc"):
        doc = read_json(f)
        repeated = _detect_repeated_lines(doc["pages"])
        clean_pages = [{"page": p["page"], "text": clean_page(p["text"], repeated)}
                       for p in doc["pages"]]
        full_text = "\n\n".join(p["text"] for p in clean_pages if p["text"])

        out = {"id": doc["id"], "source": doc["source"], "title": doc["title"],
               "category": doc["category"], "n_pages": doc["n_pages"],
               "pages": clean_pages,
               "cleaned_at": datetime.now().isoformat(timespec="seconds")}
        write_json(DIRS["cleaned"] / f"{doc['id']}.json", out)
        (DIRS["cleaned"] / f"{doc['id']}.txt").write_text(full_text, encoding="utf-8")

    log.info("Stage 3 complete. Cleaned .txt + .json in: %s", DIRS["cleaned"])


# ---- RUN STAGE 3 ----
run_cleaning()

## Cell 7 - Stage 4: Chunking

Splits cleaned text into semantically useful chunks using a recursive splitter (respects paragraph
and sentence boundaries), with configurable `chunk_size` and `chunk_overlap`.

Output: a single **`chunks.jsonl`** - one JSON object per line with metadata
`source, title, category, page, chunk_index`. The file is rebuilt fresh each run so it always
matches the current cleaned set (no stale or duplicated chunks).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def make_splitter():
    return RecursiveCharacterTextSplitter(
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"],
        separators=["\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""],
        length_function=len,
    )


def run_chunking():
    """Stage 4: chunk every cleaned document into chunks.jsonl with metadata."""
    cleaned_files = sorted(DIRS["cleaned"].glob("*.json"))
    if not cleaned_files:
        log.error("No cleaned files found. Run Stage 3 first.")
        return

    splitter = make_splitter()
    out_path = DIRS["chunked"] / "chunks.jsonl"
    n_chunks = 0

    # Rebuilt fresh so the file always matches the current cleaned set.
    with out_path.open("w", encoding="utf-8") as fout:
        for f in tqdm(cleaned_files, desc="Chunking", unit="doc"):
            doc = read_json(f)
            for page in doc["pages"]:
                if not page["text"].strip():
                    continue
                for piece in splitter.split_text(page["text"]):
                    piece = piece.strip()
                    if len(piece) < 30:               # drop tiny fragments
                        continue
                    record = {
                        "chunk_id": f"{doc['id']}_{n_chunks}",
                        "doc_id": doc["id"],
                        "source": doc["source"],
                        "title": doc["title"],
                        "category": doc["category"],
                        "page": page["page"],
                        "chunk_index": n_chunks,
                        "text": piece,
                    }
                    fout.write(json.dumps(record, ensure_ascii=False) + "\n")
                    n_chunks += 1

    log.info("Stage 4 complete. %d chunks written to %s", n_chunks, out_path)


# ---- RUN STAGE 4 ----
run_chunking()

## Cell 8 - Stage 5: Vector store (FAISS)

Builds a **persistent FAISS index** from `chunks.jsonl` using a multilingual embedding model
(`paraphrase-multilingual-MiniLM-L12-v2`) that handles Indonesian and English well.

- Index + chunk metadata are saved to `index/` on Drive and reloaded with `load_index()`.
- Embeddings are L2-normalised and the index uses inner product, giving cosine similarity.
- Rebuild only when chunks change; otherwise the cell just reloads the saved index.

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

INDEX_FILE = DIRS["index"] / "faiss.index"
META_FILE  = DIRS["index"] / "chunks_meta.json"

_embedder = None  # lazy singleton


def get_embedder():
    global _embedder
    if _embedder is None:
        log.info("Loading embedding model: %s", CONFIG["embedding_model"])
        _embedder = SentenceTransformer(CONFIG["embedding_model"])
    return _embedder


def embed_texts(texts):
    """Encode texts to L2-normalised float32 vectors (cosine similarity ready)."""
    vecs = get_embedder().encode(
        texts, batch_size=CONFIG["embedding_batch_size"],
        show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    return vecs.astype("float32")


def build_index():
    """Stage 5: build and persist the FAISS index from chunks.jsonl."""
    chunks_path = DIRS["chunked"] / "chunks.jsonl"
    if not chunks_path.exists():
        log.error("chunks.jsonl not found. Run Stage 4 first.")
        return

    records = [json.loads(ln) for ln in
               chunks_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    if not records:
        log.error("No chunks to index.")
        return
    log.info("Stage 5: embedding %d chunks...", len(records))

    vectors = embed_texts([r["text"] for r in records])
    index = faiss.IndexFlatIP(vectors.shape[1])       # inner product on normalised = cosine
    index.add(vectors)

    faiss.write_index(index, str(INDEX_FILE))
    write_json(META_FILE, records)                    # metadata aligned with index order
    log.info("Stage 5 complete. Index (%d vectors, dim=%d) saved to %s",
             index.ntotal, vectors.shape[1], DIRS["index"])


def load_index():
    """Reload a persisted index + metadata from Drive."""
    if not INDEX_FILE.exists() or not META_FILE.exists():
        log.error("No saved index found. Run build_index() first.")
        return None, None
    index = faiss.read_index(str(INDEX_FILE))
    meta = read_json(META_FILE)
    log.info("Index loaded: %d vectors, %d metadata records.", index.ntotal, len(meta))
    return index, meta


# ---- RUN STAGE 5 ----
# Build only if missing; otherwise reload (avoids re-embedding on every rerun).
if INDEX_FILE.exists() and META_FILE.exists():
    log.info("Existing index found - reloading. Call build_index() to rebuild.")
    faiss_index, chunks_meta = load_index()
else:
    build_index()
    faiss_index, chunks_meta = load_index()

## Cell 9 - Stage 6: RAG chain + evaluation

Sets up retrieval + answer generation and a basic evaluation harness.

- `retrieve()` - top-k semantic search over the FAISS index.
- `answer()` - builds a grounded answer from retrieved contexts. With an LLM API key it uses the
  LLM; without one it returns an **extractive answer** so the notebook runs with no keys.
- `evaluate()` - runs test questions, saves retrieved contexts + answers to `eval/`, and reports
  basic **retrieval quality** (keyword hit-rate, mean similarity) and **response quality**
  (answer length, context grounding overlap).

Edit `TEST_QUESTIONS` for your thesis evaluation set.

In [ ]:
def retrieve(query, top_k=None):
    """Semantic top-k retrieval. Returns chunk metadata + similarity score."""
    if faiss_index is None:
        log.error("No index loaded. Run Stage 5.")
        return []
    top_k = top_k or CONFIG["top_k"]
    qvec = embed_texts([query])
    scores, idxs = faiss_index.search(qvec, top_k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx < 0:
            continue
        rec = dict(chunks_meta[idx])
        rec["score"] = float(score)
        hits.append(rec)
    return hits


def answer(query, top_k=None, llm_api_key=None):
    """Retrieve contexts and produce a grounded answer.

    With `llm_api_key`: uses an LLM. Without it: returns a safe extractive
    answer so the pipeline is fully runnable offline (no API keys needed).
    """
    contexts = retrieve(query, top_k)
    context_block = "\n\n".join(
        f"[{i+1}] ({c['title']}, p.{c['page']}) {c['text']}"
        for i, c in enumerate(contexts))

    if llm_api_key:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=llm_api_key)
            prompt = ("Jawab pertanyaan HANYA berdasarkan konteks di bawah. "
                      "Jika tidak ada di konteks, katakan tidak tahu.\n\n"
                      f"Konteks:\n{context_block}\n\nPertanyaan: {query}")
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1)
            text = resp.choices[0].message.content
        except Exception as e:
            log.warning("LLM call failed (%s) - falling back to extractive answer.", e)
            text = contexts[0]["text"] if contexts else "Tidak ditemukan konteks relevan."
    else:
        # Extractive fallback: return the highest-scoring chunk.
        text = (contexts[0]["text"] if contexts
                else "Tidak ditemukan konteks relevan untuk pertanyaan ini.")

    return {"query": query, "answer": text,
            "contexts": [{"title": c["title"], "page": c["page"],
                          "score": c["score"], "text": c["text"]} for c in contexts]}


def evaluate(test_questions, top_k=None, llm_api_key=None):
    """Run questions, save outputs, report retrieval + response quality metrics.

    test_questions: list of {"question": str, "keywords": [str, ...]}
      `keywords` are expected terms used for a simple retrieval hit-rate check.
    """
    top_k = top_k or CONFIG["top_k"]
    results, retr_hit, retr_sim, resp_len, grounding = [], [], [], [], []

    for q in tqdm(test_questions, desc="Evaluating", unit="q"):
        res = answer(q["question"], top_k=top_k, llm_api_key=llm_api_key)
        joined = " ".join(c["text"].lower() for c in res["contexts"])

        kws = [k.lower() for k in q.get("keywords", [])]
        hit = (sum(k in joined for k in kws) / len(kws)) if kws else None
        sim = sum(c["score"] for c in res["contexts"]) / max(len(res["contexts"]), 1)
        ans_words = set(res["answer"].lower().split())
        ctx_words = set(joined.split())
        ground = len(ans_words & ctx_words) / max(len(ans_words), 1)

        if hit is not None:
            retr_hit.append(hit)
        retr_sim.append(sim)
        resp_len.append(len(res["answer"].split()))
        grounding.append(ground)
        results.append({**res, "keyword_hit_rate": hit})

    def avg(xs):
        return round(sum(xs) / len(xs), 4) if xs else None

    report = {
        "n_questions": len(test_questions),
        "top_k": top_k,
        "retrieval_quality": {
            "mean_keyword_hit_rate": avg(retr_hit),
            "mean_similarity": avg(retr_sim),
        },
        "response_quality": {
            "mean_answer_length_words": avg(resp_len),
            "mean_context_grounding": avg(grounding),
        },
        "evaluated_at": datetime.now().isoformat(timespec="seconds"),
    }
    write_json(DIRS["eval"] / "rag_results.json", results)
    write_json(DIRS["eval"] / "rag_report.json", report)
    log.info("Stage 6 complete. Results + report saved to %s", DIRS["eval"])
    return report


# --- Test set: edit for your thesis evaluation ---
TEST_QUESTIONS = [
    {"question": "Apa saja layanan informasi publik yang disediakan PPID UPI?",
     "keywords": ["informasi", "publik", "layanan"]},
    {"question": "Bagaimana prosedur pendaftaran mahasiswa baru di UPI?",
     "keywords": ["pendaftaran", "mahasiswa", "baru"]},
    {"question": "Apa fokus penelitian dan pengabdian masyarakat LPPM UPI?",
     "keywords": ["penelitian", "pengabdian", "masyarakat"]},
]

# ---- RUN STAGE 6 ----
# Set LLM_API_KEY = "sk-..." to use an LLM; leave None for offline extractive mode.
LLM_API_KEY = None

# Quick single-query smoke test:
demo = answer("Apa itu PPID UPI?", llm_api_key=LLM_API_KEY)
print("Q:", demo["query"])
print("A:", demo["answer"][:400])
print("Retrieved", len(demo["contexts"]), "contexts.\n")

# Full evaluation:
report = evaluate(TEST_QUESTIONS, llm_api_key=LLM_API_KEY)
print(json.dumps(report, ensure_ascii=False, indent=2))

---
### Re-running individual stages

Each stage reads only files on Drive, so you can run any stage on its own once its inputs exist:

- **Stage 1** -> `build_manifest(do_crawl=False)`
- **Stage 2** -> `run_extraction()`
- **Stage 3** -> `run_cleaning()`
- **Stage 4** -> `run_chunking()`
- **Stage 5** -> `build_index()` (or `load_index()` to just reload)
- **Stage 6** -> `evaluate(TEST_QUESTIONS)` or `answer("your question")`

After the first run, always run **Cell 2 + Cell 3** first to restore imports and config, then call
the stage function you need. To add a new dataset, append its path to `CONFIG["dataset_roots"]`
and re-run from Stage 1.